# Modèle de prédiction du risque de diabète

Notebook d'entraînement — équipe Data Science.

Objectif : à partir d'un profil patient (grossesses, glycémie, tension, IMC, etc.), prédire la présence d'un risque de diabète.

**Statut : pipeline entraîné et validé par validation croisée. Rien n'est packagé ni déployé, et aucun suivi de performance n'existe — c'est l'objet du TP Module 5.**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
import joblib

df = pd.read_csv("../data/raw/diabetes_train.csv")
df.head()

## 1. Description des données

| Colonne | Description |
|---|---|
| pregnancies | Nombre de grossesses |
| glucose | Glycémie plasmatique (test de tolérance au glucose) |
| blood_pressure | Tension artérielle diastolique (mm Hg) |
| skin_thickness | Épaisseur du pli cutané tricipital (mm) |
| insulin | Insuline sérique à 2h (mu U/ml) |
| bmi | Indice de masse corporelle |
| diabetes_pedigree | Fonction pedigree du diabète (facteur héréditaire) |
| age | Âge du patient |
| outcome | 1 = risque de diabète présent, 0 = absent |

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
df["outcome"].value_counts(normalize=True)

## 2. Déséquilibre de classes

~30% de cas positifs seulement. Un modèle entraîné naïvement a tendance à privilégier la classe majoritaire, ce qui est inacceptable ici : en contexte médical, **un faux négatif (rater un patient à risque) coûte bien plus qu'un faux positif**. Deux leviers sont combinés :

- **Sous-échantillonnage aléatoire** (`RandomUnderSampler`) de la classe majoritaire dans le pipeline d'entraînement, pour rééquilibrer les classes — appliqué uniquement sur le jeu d'entraînement de chaque fold, jamais sur le jeu de test. On préfère cette approche à une génération d'exemples synthétiques (type SMOTE) : sur un jeu de données cliniques, interpoler entre patients réels peut créer des profils physiologiquement peu plausibles et introduire un biais difficile à auditer. Le sous-échantillonnage ne fabrique aucune donnée : il se contente de réduire la classe majoritaire, au prix d'une partie de l'information (compensé ici par un jeu d'entraînement de taille suffisante).
- **Le rappel comme métrique d'optimisation** de la recherche d'hyperparamètres, plutôt que l'accuracy, qui serait trompeuse sur des classes déséquilibrées.

## 3. Séparation train / test

In [ ]:
FEATURES = [
    "pregnancies", "glucose", "blood_pressure", "skin_thickness",
    "insulin", "bmi", "diabetes_pedigree", "age",
]

X = df[FEATURES]
y = df["outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

## 4. Pipeline de prétraitement et de modélisation

Le pipeline enchaîne : imputation des valeurs manquantes (médiane, par robustesse même si ce jeu n'en comporte pas), standardisation, sous-échantillonnage de la classe majoritaire, puis le classifieur. Le tout est encapsulé dans un seul objet scikit-learn/imblearn, pour garantir que le même prétraitement est appliqué à l'entraînement et à l'inférence — et que le ré-échantillonnage ne s'applique jamais au moment de la prédiction (comportement natif d'un `imblearn.pipeline.Pipeline`).

In [ ]:
pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("undersampler", RandomUnderSampler(random_state=42)),
    ("model", RandomForestClassifier(random_state=42)),
])
pipeline

## 5. Recherche d'hyperparamètres

`GridSearchCV` avec validation croisée stratifiée (5 folds), optimisée sur le **rappel**. La grille reste volontairement restreinte (démonstration pédagogique) ; en production on élargirait la recherche ou on passerait à une recherche aléatoire/bayésienne.

In [ ]:
param_grid = {
    "model__n_estimators": [200, 300, 400],
    "model__max_depth": [4, 6, 8, None],
    "model__min_samples_leaf": [1, 3, 5],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(pipeline, param_grid=param_grid, scoring="recall", cv=cv, n_jobs=-1)
search.fit(X_train, y_train)

search.best_params_, round(search.best_score_, 4)

## 6. Évaluation du meilleur modèle

In [ ]:
best_pipeline = search.best_estimator_

y_pred = best_pipeline.predict(X_test)
y_proba = best_pipeline.predict_proba(X_test)[:, 1]

print("accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("recall:  ", round(recall_score(y_test, y_pred), 4))
print("f1:      ", round(f1_score(y_test, y_pred), 4))
print("auc:     ", round(roc_auc_score(y_test, y_proba), 4))
print("confusion matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
importances = pd.Series(
    best_pipeline.named_steps["model"].feature_importances_, index=FEATURES
).sort_values(ascending=False)
importances

Les variables les plus discriminantes restent cohérentes avec la littérature clinique : `glucose`, `bmi`, `age`, `diabetes_pedigree`.

## 7. Sauvegarde du pipeline

Le pipeline complet (prétraitement + modèle) est sauvegardé en `.pkl`, avec la liste des features attendues. **À partir d'ici, ce n'est plus le travail de la data scientist : le suivi en production (MLflow, monitoring, détection de dérive) et l'intégration continue (CI/CD) sont à la charge de l'équipe MLOps (TP Module 5).**

In [ ]:
joblib.dump({"model": best_pipeline, "features": FEATURES}, "../diabetes_risk_model.pkl")